# 03 — Baselines and teachers (go / no-go)

Implements `notebooks/03_spec.md` (source of truth). Logic lives in `hebrew_token_tax.test3`,
`compressors` and `llm`; this notebook only configures, calls and shows. Guide: `03_README.md`.

## Run plan (change only the flags in cell 2, then *Run All*)

| Step | What | Flags (everything else as default) | Cost |
|---|---|---|---|
| 1 | Offline check (optional) | `PROVIDER_OVERRIDE = "fake"`, `RUN_LOCAL = RUN_TEACHERS = RUN_ANSWERS = RUN_JUDGE = True` | free |
| 2 | Smoke test (verifies the 4 model ids) | `RUN_SMOKE = True`, `CONFIRM_SPEND = True` | ~cents |
| 3 | Dry-run estimate (3 cases) | `RUN_LOCAL = RUN_TEACHERS = RUN_ANSWERS = RUN_JUDGE = True`, `CONFIRM_SPEND = False` | free (stops at the teacher estimate) |
| 4 | Dry run (3 cases) | as step 3, `CONFIRM_SPEND = True` | small |
| 5 | Full-run estimate (48 cases) | as step 3, `DRY_RUN_N = None` | free (local stage ~25 min CPU) |
| 6 | Full run | as step 5, `CONFIRM_SPEND = True` | paid |

After step 4 check the last cell: spend, failures, and the output tokens used per role.
Paid stages print *calls to make / cached / estimated cost* and stop with `SpendBlocked`
unless `CONFIRM_SPEND = True`. If a stage stops midway (`BillingError`, `FatalCallError`,
`StageAborted`, `SpendBlocked`), read the message, fix the cause and run again: finished
calls come from the cache and are never paid twice.

### 1. Imports and prompts — loads the code and prints the 4 prompts verbatim from the spec. [FREE]

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv

from hebrew_token_tax.compressors import (
    LLMLINGUA2_LARGE,
    LLMLINGUA2_SMALL,
    LLMLingua2Compressor,
    NllbTranslator,
    RuleCompressor,
)
from hebrew_token_tax.llm import (
    AnthropicProvider,
    FakeProvider,
    GeminiProvider,
    LLMClient,
    OpenAIProvider,
    Price,
    guard_spend,
)
from hebrew_token_tax.test3 import (
    LANGUAGES,
    JsonStore,
    LocalCompressors,
    Role,
    answer_jobs,
    answer_requests,
    context,
    fake_reply,
    judge_jobs,
    judge_requests,
    load_cases,
    load_prompts,
    local_pending,
    method_grid,
    pending_teacher_jobs,
    plot_savings_vs_accuracy,
    resolve_roles,
    run_answers,
    run_judge,
    run_local,
    run_teachers,
    score_rows,
    smoke_requests,
    summarize,
    teacher_request,
    token_ratio,
    write_summary_md,
)

SPEC_PATH = Path("03_spec.md")
PROMPTS = load_prompts(SPEC_PATH)
for name, prompt in PROMPTS.items():
    print(f"===== {name} =====\n{prompt}\n")

### 2. Config — models, output limits, prices, budget caps and run flags. [FREE]

- Output limits include reasoning tokens and are only a ceiling (only used tokens are billed).
  Teachers 16k; chatbot 4k and judge 8k leave room for low/medium reasoning plus the short
  JSON. A reply that hits its limit is marked failed. The last cell shows the tokens actually
  used per role, so the limits can be tightened after the dry run.
- `DEFAULT_OUTPUT_TOKENS` only feeds the cost estimate until real usage is cached.
- `DRY_RUN_N = 3` = the first 3 cases of each language (same ids), reused from cache later.

In [ ]:
TEACHER_SOL_MODEL = "gpt-6.1-sol"
TEACHER_OPUS_MODEL = "claude-opus-5-5"
CHATBOT_MODEL = "gpt-6.1-terra"
JUDGE_MODEL = "claude-sonnet-5-5"
TEACHER_MAX_OUTPUT_TOKENS = 16000
CHATBOT_MAX_OUTPUT_TOKENS = 4000
JUDGE_MAX_OUTPUT_TOKENS = 8000

ROLES = {
    "teacher_sol": Role("openai", TEACHER_SOL_MODEL, "high", TEACHER_MAX_OUTPUT_TOKENS),
    "teacher_opus": Role("anthropic", TEACHER_OPUS_MODEL, "high", TEACHER_MAX_OUTPUT_TOKENS),
    "chatbot": Role("openai", CHATBOT_MODEL, "low", CHATBOT_MAX_OUTPUT_TOKENS),
    "judge": Role("anthropic", JUDGE_MODEL, "medium", JUDGE_MAX_OUTPUT_TOKENS),
}
PRICES = {
    TEACHER_SOL_MODEL: Price(2.0, 10.0),
    TEACHER_OPUS_MODEL: Price(4.0, 20.0),
    CHATBOT_MODEL: Price(1.0, 5.0),
    JUDGE_MODEL: Price(2.0, 10.0),
}
BUDGET_CAPS = {"anthropic": 25.0, "openai": 15.0}
DEFAULT_OUTPUT_TOKENS = {
    "smoke": 50,
    "teacher_sol": 3000,
    "teacher_opus": 3000,
    "chatbot": 800,
    "judge": 1500,
}

KEEP_RATES = (0.7, 0.5)
DRY_RUN_N = 3
PROVIDER_OVERRIDE = None
DEBUG_MODELS = {"gemini": "gemini-3.5-flash-lite", "fake": "fake"}
GEMINI_KEY_NAMES = ("GEMINI_API_KEY", "GEMINI_API_KEY_2")
GEMINI_REQUESTS_PER_MINUTE = 15

RUN_SMOKE = False
RUN_LOCAL = False
RUN_TEACHERS = False
RUN_ANSWERS = False
RUN_JUDGE = False
CONFIRM_SPEND = False

DATA_DIR = Path("..") / "data" / "03"
RESULTS_DIR = Path("..") / "results"
CACHE_DIR = Path("..") / "data" / "cache" / "03"
MODE = PROVIDER_OVERRIDE or "real"
STAGE_DIR = CACHE_DIR / MODE
SUFFIX = "" if MODE == "real" and DRY_RUN_N is None else f"_{MODE}_n{DRY_RUN_N or 'all'}"

if not RESULTS_DIR.is_dir():
    raise FileNotFoundError(f"{RESULTS_DIR} not found; run from the notebooks/ folder")
print(f"mode: {MODE}, cases per language: {DRY_RUN_N or 'all'}, result suffix: {SUFFIX!r}")
print(f"budget caps: {BUDGET_CAPS}, CONFIRM_SPEND = {CONFIRM_SPEND}")

### 3. Environment and client — loads `.env` only when a real provider needs keys (never in fake mode), checks the key names exist (values never printed), builds the client. [FREE]

In [ ]:
KEY_NAMES = {
    "openai": ("OPENAI_API_KEY",),
    "anthropic": ("ANTHROPIC_API_KEY",),
    "gemini": GEMINI_KEY_NAMES,
    "fake": (),
}

roles = resolve_roles(ROLES, PROVIDER_OVERRIDE, DEBUG_MODELS)
providers_in_use = sorted({role.provider for role in roles.values()})
if any(KEY_NAMES[provider] for provider in providers_in_use):
    load_dotenv()
missing = [key for provider in providers_in_use for key in KEY_NAMES[provider] if not os.getenv(key)]
if missing:
    raise RuntimeError(f"Missing from .env: {', '.join(missing)}")

client = LLMClient(
    CACHE_DIR / "llm",
    providers={
        "openai": OpenAIProvider(),
        "anthropic": AnthropicProvider(),
        "gemini": GeminiProvider(GEMINI_KEY_NAMES, GEMINI_REQUESTS_PER_MINUTE),
        "fake": FakeProvider(fake_reply),
    },
    prices=PRICES,
    budget_caps=BUDGET_CAPS,
)
for name, role in roles.items():
    print(f"{name:13} {role.provider:10} {role.model:24} reasoning={role.reasoning:7} "
          f"max_output_tokens={role.max_output_tokens}")

### 4. Cases — loads the cases (first `DRY_RUN_N` per language), builds the 23 method/rate/language combos, prints the Hebrew/English token ratio. [FREE]

In [ ]:
cases = {language: load_cases(DATA_DIR, language, DRY_RUN_N) for language in LANGUAGES}
methods = method_grid(KEEP_RATES)
ratio = token_ratio(cases)

print(f"cases: {len(cases['he'])} he, {len(cases['en'])} en; {len(methods)} method/rate/language combos")
print(f"ids: {[case['id'] for case in cases['he']][:5]}{' ...' if len(cases['he']) > 5 else ''}")
print(
    f"Hebrew/English o200k ratio of originals: {ratio['overall']:.2f} overall "
    f"({ratio['he_tokens']} / {ratio['en_tokens']}), {ratio['mean_per_case']:.2f} mean per case"
)

### 5. Smoke test — one tiny "Reply with OK" call per model in use (`RUN_SMOKE`). [PAID]

In [ ]:
if RUN_SMOKE:
    smoke = smoke_requests(roles)
    guard_spend(client, client.estimate(smoke, DEFAULT_OUTPUT_TOKENS), CONFIRM_SPEND, "smoke")
    for result in client.call_many(smoke, desc="smoke"):
        request = result.request
        print(
            f"{request.provider:10} {request.model:24} reply={result.text[:40]!r} "
            f"error={result.error or '-'} cost=${result.cost_usd:.5f} cached={result.cached}"
        )
else:
    print("smoke test skipped (RUN_SMOKE = False)")

### 6. Local methods — rules, LLMLingua-2 large/small, NLLB, NLLB + LLMLingua-2 on CPU; models load only if something is pending (`RUN_LOCAL`). [FREE]

In [ ]:
local = JsonStore(CACHE_DIR / "local.json")
pending = local_pending(cases, methods, local)
if RUN_LOCAL and pending:
    tools = LocalCompressors(
        rules={language: RuleCompressor(language, DATA_DIR) for language in LANGUAGES},
        llmlingua2_large=LLMLingua2Compressor(LLMLINGUA2_LARGE),
        llmlingua2_small=LLMLingua2Compressor(LLMLINGUA2_SMALL),
        translator=NllbTranslator(),
    )
    tools.warm_up(context(cases["he"][0]))
    print(f"compressed {run_local(cases, methods, tools, local)} texts")
else:
    print(f"local stage skipped (RUN_LOCAL = {RUN_LOCAL}, pending = {pending})")
print(f"stored local compressions: {len(local.data)}")

### 7. Teachers — two LLMs shorten each text by deletion only; invalid output is retried up to 2 times, then the original is kept and marked failed (`RUN_TEACHERS`). [PAID]

In [ ]:
teachers = JsonStore(STAGE_DIR / "teachers.json")
if RUN_TEACHERS:
    jobs = pending_teacher_jobs(cases, methods, teachers)
    first_attempts = [teacher_request(roles, PROMPTS, case, method, 0) for case, method in jobs]
    guard_spend(client, client.estimate(first_attempts, DEFAULT_OUTPUT_TOKENS), CONFIRM_SPEND, "teachers")
    print(f"teacher failures this run: {run_teachers(client, roles, PROMPTS, jobs, teachers)}")
failed = sum(record["teacher_failed"] for record in teachers.data.values())
print(f"stored teacher compressions: {len(teachers.data)}, failed: {failed}")

### 8. Chatbot answers — answers the 3 questions from each text as JSON; invalid JSON is retried once, then counted as 3 incorrect (`RUN_ANSWERS`). [PAID]

In [ ]:
compressions = JsonStore.combined(local, teachers)
answers = JsonStore(STAGE_DIR / "answers.json")
if RUN_ANSWERS:
    jobs = answer_jobs(cases, methods, compressions, answers)
    requests = answer_requests(roles, PROMPTS, jobs, compressions)
    guard_spend(client, client.estimate(requests, DEFAULT_OUTPUT_TOKENS), CONFIRM_SPEND, "answers")
    print(f"answer JSON failures this run: {run_answers(client, roles, PROMPTS, jobs, compressions, answers)}")
json_failed = sum(record["json_failed"] for record in answers.data.values())
print(f"stored answer sets: {len(answers.data)}, JSON failures: {json_failed}")

### 9. Judge — one blind call per answer set grades the 3 answers against the correct answers; invalid JSON counts as incorrect (`RUN_JUDGE`). [PAID]

In [ ]:
judgements = JsonStore(STAGE_DIR / "judgements.json")
if RUN_JUDGE:
    jobs = judge_jobs(cases, methods, answers, judgements)
    requests = judge_requests(roles, PROMPTS, jobs, answers)
    guard_spend(client, client.estimate(requests, DEFAULT_OUTPUT_TOKENS), CONFIRM_SPEND, "judge")
    print(f"judge JSON failures this run: {run_judge(client, roles, PROMPTS, jobs, answers, judgements)}")
json_failed = sum(record["json_failed"] for record in judgements.data.values())
print(f"stored judgements: {len(judgements.data)}, JSON failures: {json_failed}")

### 10. Metrics — one row per case/method/rate/language (no text) and the means; writes `results/03_results*.csv` and `03_summary*.csv`. [FREE]

In [ ]:
rows = score_rows(cases, methods, compressions, answers, judgements)
summary = summarize(rows)

results_csv = RESULTS_DIR / f"03_results{SUFFIX}.csv"
summary_csv = RESULTS_DIR / f"03_summary{SUFFIX}.csv"
rows.to_csv(results_csv, index=False)
summary.to_csv(summary_csv, index=False)
print(f"{len(rows)} rows -> {results_csv}\n{len(summary)} rows -> {summary_csv}\n")
print(summary.round(1).to_string(index=False))

### 11. Chart — mean token savings vs. accuracy, one panel per language; writes `results/03_savings_vs_accuracy*.png`. [FREE]

In [ ]:
chart_path = RESULTS_DIR / f"03_savings_vs_accuracy{SUFFIX}.png"
plot_savings_vs_accuracy(summary, chart_path)
plt.show()

### 12. Summary file — token ratio, tables and chart (no verdict) in `results/03_summary*.md`. [FREE]

In [ ]:
summary_md = RESULTS_DIR / f"03_summary{SUFFIX}.md"
write_summary_md(
    summary_md,
    summary,
    ratio,
    chart_path.name,
    {language: len(cases[language]) for language in LANGUAGES},
)
print(f"-> {summary_md}")

### 13. Spend and usage — calls and cost this session, all-time spend vs. caps, and output tokens used per role vs. its limit. [FREE]

In [ ]:
print(f"this session: calls {dict(client.ledger.calls)}")
for provider, cost in sorted(client.ledger.cost.items()):
    print(f"  {provider}: ${cost:.4f}")
all_time = client.all_time_spend()
print(f"all time: ${sum(all_time.values()):.4f}")
for provider, cost in sorted(all_time.items()):
    cap = BUDGET_CAPS.get(provider)
    print(f"  {provider}: ${cost:.4f}" + (f" of ${cap:.2f} cap" if cap else ""))
print()
print(pd.DataFrame(client.usage_by_role()).to_string(index=False))